# ZODIAK AI — vidéos de la vitrine (v2)
Modèle : **Wan 2.2 T2V A14B** (Apache-2.0), quantifié GGUF Q4_K_M, accéléré par **Wan2.2-Lightning** (4 étapes).
Réglages Kaggle : *Accelerator* = **GPU T4 x2** (ou P100), *Internet* = **On**. Puis **Run All**.
Chaque clip terminé est écrit dans `/kaggle/working/out/` : si la session s'arrête, relancer reprend où elle en était.
À la fin, télécharger **`zodiak-vitrine.zip`** dans l'onglet *Output*.


In [ ]:
# 1. Installation (2 à 3 min)
!pip install -q -U "diffusers>=0.35" transformers accelerate gguf ftfy peft imageio imageio-ffmpeg sentencepiece


In [ ]:
# 2. Liste des clips (générée depuis scripts/showcase-clips.mjs)
CLIPS = [
 {
  "id": "c01",
  "title": "Kickflip au couchant",
  "portrait": false,
  "prompt": "A young skateboarder in a red hoodie mid-kickflip on an empty concrete plaza at sunset, low angle, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Fast crash zoom in on the skateboarder as the board flips under his feet, warm sun flare, film grain."
 },
 {
  "id": "c02",
  "title": "Canyon en FPV",
  "portrait": false,
  "prompt": "A dirt bike rider kicking up dust in a narrow red rock desert canyon, aerial view, cinematic film still, no logo, no text. FPV drone dives into the canyon and chases the rider at high speed, dust trails, fast smooth motion."
 },
 {
  "id": "c03",
  "title": "Danse dans la lumière",
  "portrait": true,
  "prompt": "A contemporary dancer in a flowing white dress spinning in an empty warehouse, shafts of light and floating dust, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Camera orbits around the dancer as she spins, the dress flows, light beams shimmer through the dust."
 },
 {
  "id": "c04",
  "title": "Dernier round",
  "portrait": false,
  "prompt": "A boxer with taped hands in a dim old gym ring, sweat on his face, harsh top light, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Handheld camera shakes as the boxer throws fast punches toward the lens, sweat droplets fly."
 },
 {
  "id": "c05",
  "title": "Défilé sous la pluie",
  "portrait": true,
  "prompt": "A fashion model in a long black coat walking down a rainy street at night, neon reflections on wet asphalt, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Camera tracks backward as she walks confidently toward the lens, rain falls, neon lights reflect."
 },
 {
  "id": "c06",
  "title": "Drift de nuit",
  "portrait": false,
  "prompt": "A matte black sports car drifting on a wet race track at night, thick tire smoke, stadium lights, generic car without badge, cinematic film still. Camera tracks sideways with the drifting car, tire smoke billows, rain spray in the lights."
 },
 {
  "id": "c07",
  "title": "Ruines de jungle",
  "portrait": false,
  "prompt": "A small weathered explorer robot standing in overgrown jungle temple ruins, morning mist, god rays, cinematic film still, no logo. Crane shot rising from the robot to reveal the huge ruins and the canopy, birds fly away."
 },
 {
  "id": "c08",
  "title": "Toit à l’heure dorée",
  "portrait": true,
  "prompt": "A woman with freckles and wind-blown hair on a city rooftop at golden hour, warm backlight, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Slow push-in on her face, she turns toward the camera with a slight smile, hair moves in the wind."
 },
 {
  "id": "c09",
  "title": "Dans le tube",
  "portrait": false,
  "prompt": "A surfer riding inside a turquoise barrel wave, sunlight through the water, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Camera tracks alongside the surfer inside the barrel, spray glitters, water curls overhead."
 },
 {
  "id": "c10",
  "title": "Flambé",
  "portrait": false,
  "prompt": "Close-up of a chef flambéing in a steel pan in a dark restaurant kitchen, burst of orange flames, cinematic film still, no logo. Macro close-up, flames burst from the pan in slow motion, oil sizzles, sparks rise."
 },
 {
  "id": "c11",
  "title": "Immobile dans la foule",
  "portrait": false,
  "prompt": "A busy city crossing at night seen from above, one person in a yellow raincoat standing still among the crowd, cinematic film still, no text. Time-lapse from above, the crowd and car lights blur around the still figure."
 },
 {
  "id": "c12",
  "title": "Sortie du hangar",
  "portrait": false,
  "prompt": "An astronaut walking out of a massive spaceship hangar onto an icy planet, blowing snow, huge scale, generic suit without insignia, cinematic film still. Wide establishing shot, the astronaut walks forward, snow blows across the ground, hangar lights flicker."
 },
 {
  "id": "c13",
  "title": "Dunk au crépuscule",
  "portrait": false,
  "prompt": "A basketball player rising for a dunk on an outdoor court at dusk, low angle, generic jersey without logo, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Low angle slow motion as the player rises and slams the ball, the net swings."
 },
 {
  "id": "c14",
  "title": "Tunnel orange",
  "portrait": false,
  "prompt": "A motorcycle rider in a black helmet speeding through a tunnel of orange lights, anamorphic lens flares, generic bike without logo, cinematic film still. Fast forward motion following the rider, light streaks rush past, strong sense of speed."
 },
 {
  "id": "c15",
  "title": "Porte du château",
  "portrait": false,
  "prompt": "A knight in dark armor standing at a castle gate at night in the rain, burning torches, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Slow pull back revealing the gate and the soldiers behind him, rain falls, torches flicker."
 },
 {
  "id": "c16",
  "title": "Solo sur scène",
  "portrait": true,
  "prompt": "A guitarist playing an electric guitar on a smoky stage, strong colored backlight, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Locked-off shot, the guitarist plays energetically, stage lights sweep through the smoke."
 },
 {
  "id": "c17",
  "title": "Galop à l’aube",
  "portrait": false,
  "prompt": "A rider on a galloping horse along a beach at dawn, pastel sky, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Camera pans and tracks alongside the horse, sand and water splash in slow motion."
 },
 {
  "id": "c18",
  "title": "Ruelle aux lanternes",
  "portrait": true,
  "prompt": "A young woman riding a bicycle through a narrow alley lit by paper lanterns at night, cinematic film still, 35mm, natural skin texture, shallow depth of field, an original fictional person, no logo, no text. Handheld follow shot, she rides past the lanterns, hair and scarf move in the wind."
 },
 {
  "id": "c19",
  "title": "Éclat de verre",
  "portrait": true,
  "prompt": "A clear glass perfume bottle without any logo on a black reflective surface, studio lighting, product photography. Extreme slow motion, water droplets explode around the bottle and catch the light."
 },
 {
  "id": "c20",
  "title": "Saut en poudreuse",
  "portrait": false,
  "prompt": "A snowboarder launching off a powder ridge above snowy peaks, bright sun, generic gear without logo, cinematic film still. Drone follows the snowboarder off the ridge, powder sprays, sun flare."
 }
]
NEGATIVE = "blurry, low quality, distorted face, deformed hands, extra limbs, text, watermark, logo, static image, flicker"
TEST_ONLY = False   # True : un seul clip court pour vérifier que tout marche


In [ ]:
# 3a. Textes -> vecteurs (UMT5 sur CPU en bfloat16 : en float16 il déborde ; ≈ 1 min par prompt)
import gc, os, glob, torch
os.environ.setdefault('HF_HOME', '/tmp/hf')
from huggingface_hub import snapshot_download, hf_hub_download
from diffusers import WanPipeline, WanTransformer3DModel, AutoencoderKLWan, GGUFQuantizationConfig
from diffusers.utils import export_to_video
# Seulement les petits dossiers : les poids complets (≈ 120 Go) sont remplacés par les GGUF.
BASE = snapshot_download('Wan-AI/Wan2.2-T2V-A14B-Diffusers', allow_patterns=['*.json', 'tokenizer/*', 'text_encoder/*', 'vae/*', 'scheduler/*'])
enc = WanPipeline.from_pretrained(BASE, transformer=None, transformer_2=None, vae=None, torch_dtype=torch.bfloat16)
EMB = {}
with torch.no_grad():
    for c in (CLIPS[:1] if TEST_ONLY else CLIPS):
        e, _ = enc.encode_prompt(c['prompt'], do_classifier_free_guidance=False, device='cpu', dtype=torch.bfloat16)
        EMB[c['id']] = e.to(torch.float16)
        print('texte prêt', c['id'])
del enc; gc.collect()


In [ ]:
# 3b. Modèle vidéo : Wan 2.2 A14B GGUF Q4_K_M + Lightning 4 étapes
DT = torch.float16  # T4/P100 : pas de bfloat16 natif
GGUF = 'QuantStack/Wan2.2-T2V-A14B-GGUF'
q = GGUFQuantizationConfig(compute_dtype=DT)
t1 = WanTransformer3DModel.from_single_file(hf_hub_download(GGUF, 'HighNoise/Wan2.2-T2V-A14B-HighNoise-Q4_K_M.gguf'), quantization_config=q, config=BASE, subfolder='transformer', torch_dtype=DT)
t2 = WanTransformer3DModel.from_single_file(hf_hub_download(GGUF, 'LowNoise/Wan2.2-T2V-A14B-LowNoise-Q4_K_M.gguf'), quantization_config=q, config=BASE, subfolder='transformer_2', torch_dtype=DT)
vae = AutoencoderKLWan.from_pretrained(BASE, subfolder='vae', torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(BASE, transformer=t1, transformer_2=t2, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=DT)
LORA, LDIR = 'lightx2v/Wan2.2-Lightning', 'Wan2.2-T2V-A14B-4steps-lora-250928'
pipe.load_lora_weights(LORA, weight_name=f'{LDIR}/high_noise_model.safetensors', adapter_name='hi')
pipe.load_lora_weights(LORA, weight_name=f'{LDIR}/low_noise_model.safetensors', adapter_name='lo', load_into_transformer_2=True)
pipe.vae.enable_tiling()
pipe.enable_model_cpu_offload()
print('Modèle vidéo prêt')


In [ ]:
# 4. Génération (≈ 5 à 12 min par clip sur T4)
import time
os.makedirs('/kaggle/working/out', exist_ok=True)
todo = CLIPS[:1] if TEST_ONLY else CLIPS
for c in todo:
    path = f"/kaggle/working/out/{c['id']}.mp4"
    if os.path.exists(path):
        print('déjà fait', c['id']); continue
    w, h = (480, 832) if c['portrait'] else (832, 480)
    t0 = time.time()
    frames = pipe(prompt_embeds=EMB[c['id']].to('cuda'), width=w, height=h,
                  num_frames=33 if TEST_ONLY else 81, num_inference_steps=4, guidance_scale=1.0, guidance_scale_2=1.0,
                  generator=torch.Generator('cpu').manual_seed(1000 + int(c['id'][1:]))).frames[0]
    export_to_video(frames, path, fps=16)
    gc.collect(); torch.cuda.empty_cache()
    print(f"{c['id']} {c['title']} OK en {time.time() - t0:.0f} s")


In [ ]:
# 5. Aperçu du dernier clip
from IPython.display import Video
done = sorted(glob.glob('/kaggle/working/out/*.mp4'))
print(len(done), 'clips :', [os.path.basename(p) for p in done])
Video(done[-1], embed=True, width=480) if done else None


In [ ]:
# 6. Archive à télécharger (onglet Output)
!cd /kaggle/working && rm -f zodiak-vitrine.zip && zip -q -j zodiak-vitrine.zip out/*.mp4 && ls -lh zodiak-vitrine.zip
